**Week 3 Assignment — Leakage-Safe PreprocessingSource: UCI Machine Learning Repository (Default of Credit Card Clients Dataset).**                                                             
*   Citation & Licence: Yeh, I. (2009). Default of Credit Card Clients [Dataset]. UCI Machine Learning Repository. DOI: https://doi.org/10.24432/C55S3H. Licence: Creative Commons Attribution 4.0 International (CC BY 4.0).
*   Observation Unit: One borrower credit-card account record (30,000 εγγραφές)
*   Target: default payment next month (Binary: 1 = default, 0 = no default).   
*   Variable Meanings: Περιλαμβάνει το πιστωτικό όριο (LIMIT_BAL), δημογραφικά στοιχεία (SEX, EDUCATION, MARRIAGE, AGE), ιστορικό πληρωμών (PAY_0 έως PAY_6), ποσά λογαριασμών (BILL_AMT1–BILL_AMT6) και προηγούμενες πληρωμές (PAY_AMT1–PAY_AMT6).


  



In [2]:
# ==========================================
# 0. ΕΙΣΑΓΩΓΗ ΒΙΒΛΙΟΘΗΚΩΝ & ΦΟΡΤΩΣΗ ΔΕΔΟΜΕΝΩΝ
# ==========================================
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Φόρτωση του κανονικού αρχείου δεδομένων από το πακέτο του μαθήματος
# (Βεβαιωθείτε ότι το default_of_credit_card_clients.csv.gz βρίσκεται στον φόκελο του Colab)
df = pd.read_csv('default_of_credit_card_clients.csv.gz', compression='gzip', header=0)

# ==========================================
# 1. DATA AUDIT & DESCRIPTIVES
# ==========================================
print("=== DATA AUDIT ===")
print(f"Shape: {df.shape}")
print(f"Data Types:\n{df.dtypes}")
print(f"Missing Values (Total): {df.isnull().sum().sum()}")
print(f"Duplicate Rows Count: {df.duplicated().sum()}")
print(f"Unique ID Count: {df['ID'].nunique()} (Total rows: {len(df)})")

print("\nTarget Balance (`default payment next month`):")
print(df['default payment next month'].value_counts())
print(df['default payment next month'].value_counts(normalize=True) * 100)

print("\nCoded-Category Frequencies (e.g., EDUCATION):")
print(df['EDUCATION'].value_counts())

# ==========================================
# 2. CONTROLLED MISSINGNESS EXAMPLE (In-Memory)
# ==========================================
# Δημιουργούμε αντίγραφο για ασφάλεια και εισάγουμε ελεγχόμενα missing values στο LIMIT_BAL
df_missing = df.copy()
np.random.seed(42)
mask = np.random.rand(len(df_missing)) < 0.05  # 5% των γραμμών
df_missing.loc[mask, 'LIMIT_BAL'] = np.nan

print(f"\nControlled Missingness created. Missing LIMIT_BAL count: {df_missing['LIMIT_BAL'].isnull().sum()}")
# Επεξήγηση: Για τη διαχείριση χρησιμοποιείται συνδυασμός Imputation (π.χ. Median Imputer για τη συνεχή μεταβλητή)
# και Missing Indicator (δημιουργία δυαδικής στήλης που σηματοδοτεί αν η τιμή ήτανmissing),
# διότι η απουσία μπορεί να εμπεριέχει κρυφή πληροφορία (informative missingness).

# ==========================================
# 3. DUPLICATE-DETECTION EXAMPLE
# ==========================================
# Ελέγχουμε για διπλότυπες εγγραφές (εξαιρώντας το ID)
duplicates = df[df.duplicated(subset=['LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_0'], keep=False)]
print(f"\nPotential duplicate rows found based on features: {len(duplicates)}")
# Επεξήγηση: Ένα αναγνωριστικό (ID) δεν αποδεικνύει από μόνο του ότι δύο γραμμές είναι οικονομικά διαφορετικές,
# καθώς διαφορετικοί πελάτες (διαφορετικά IDs) μπορεί να συμπίπτουν ακριβώς στα δημογραφικά και πιστωτικά τους χαρακτηριστικά.

# ==========================================
# 4. OUTLIER FLAG (Numeric Variable)
# ==========================================
# Εφαρμογή κανόνα IQR για τον εντοπισμό outliers στο LIMIT_BAL
Q1 = df['LIMIT_BAL'].quantile(0.25)
Q3 = df['LIMIT_BAL'].quantile(0.75)
IQR = Q3 - Q1
upper_threshold = Q3 + 1.5 * IQR
lower_threshold = Q1 - 1.5 * IQR

outliers_count = df[(df['LIMIT_BAL'] > upper_threshold) | (df['LIMIT_BAL'] < lower_threshold)].shape[0]
print(f"\nOutlier Threshold for LIMIT_BAL: Upper={upper_threshold}, Lower={lower_threshold}")
print(f"Number of flagged outliers: {outliers_count}")
# Απόφαση: Τα ακραία όρια πίστωσης (high limits) είναι έγκυρες οικονομικές τιμές (π.χ. wealthy clients)
# και δεν πρέπει να αφαιρεθούν τυφλά χωρίς επιχειρηματικό λόγο.

# ==========================================
# 5. COLUMN SEPARATION (Numeric, Categorical, Identifier, Target)
# ==========================================
target_col = 'default payment next month'
id_col = 'ID'
categorical_cols = ['SEX', 'EDUCATION', 'MARRIAGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']
numeric_cols = [col for col in df.columns if col not in [target_col, id_col] + categorical_cols]

print(f"\nNumeric columns count: {len(numeric_cols)}")
print(f"Categorical columns count: {len(categorical_cols)}")

# ==========================================
# 6. TRAIN-TEST SPLIT (Fixed Seed & Stratification)
# ==========================================
X = df.drop(columns=[target_col, id_col])
y = df[target_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

# ==========================================
# 7. COLUMN TRANSFORMER & PIPELINES (Leakage-Safe)
# ==========================================
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

# Fit αποκλειστικά στα δεδομένα εκπαίδευσης (Training-only fitting) για αποφυγή data leakage
X_train_transformed = preprocessor.fit_transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

print(f"\nTransformed Training Shape: {X_train_transformed.shape}")
print(f"Transformed Testing Shape: {X_test_transformed.shape}")

=== DATA AUDIT ===
Shape: (30000, 25)
Data Types:
ID                            int64
LIMIT_BAL                     int64
SEX                           int64
EDUCATION                     int64
MARRIAGE                      int64
AGE                           int64
PAY_0                         int64
PAY_2                         int64
PAY_3                         int64
PAY_4                         int64
PAY_5                         int64
PAY_6                         int64
BILL_AMT1                     int64
BILL_AMT2                     int64
BILL_AMT3                     int64
BILL_AMT4                     int64
BILL_AMT5                     int64
BILL_AMT6                     int64
PAY_AMT1                      int64
PAY_AMT2                      int64
PAY_AMT3                      int64
PAY_AMT4                      int64
PAY_AMT5                      int64
PAY_AMT6                      int64
default payment next month    int64
dtype: object
Missing Values (Total): 0
Duplicate 

**Ερμηνεία & Αποτροπή Διαρροής Δεδομένων (Data Leakage Prevention):**

Πώς το workflow αποτρέπει τη διαρροή (Plain Language):

Η χρήση του ColumnTransformer εντός ενός Pipeline διασφαλίζει ότι οι παράμετροι μετασχηματισμού (όπως οι μέσοι όροι imputation στο SimpleImputer και οι μέσοι/τυπικές αποκλίσεις στο StandardScaler) υπολογίζονται αποκλειστικά και μόνο επί των δεδομένων εκπαίδευσης (X_train) κατά τη διάρκεια της εντολής .fit_transform(). Τα δεδομένα ελέγχου (X_test) υφίστανται απλώς τον μετασχηματισμό (.transform()) χρησιμοποιώντας τις στατιστικές τιμές του train, αποτρέποντας έτσι οποιαδήποτε «μόλυνση» (leakage) μελλοντικής ή εκτός δείγματος πληροφορίας στο μοντέλο.

Χρήση handle_unknown='ignore':

Η ρύθμιση αυτή στον OneHotEncoder διασφαλίζει ότι εάν το σύνολο ελέγχου (test set) περιέχει κάποια κατηγορία που δεν είχε εμφανιστεί ποτέ στο train set, ο encoder δεν θα προκαλέσει σφάλμα αλλά θα την αγνοήσει ασφαλώς δημιουργώντας μηδενικές στήλες για τη συγκεκριμένη κατηγορία.